# 00 · The Monthly Stats data — what it is, and how to read it

This is the starting point for the whole workshop. Afterwards you will be able to say what the Monthly Stats dataset is, what its four metrics mean, how to read a percentile, and which pieces of the data exist for your own country.

**The plan for this workshop**

| Notebook | Question it answers |
|---|---|
| **00** · this one | What are these data, and how do I load them? |
| **01** · country explorer | How do I read one country's internet from its distribution shape? |
| **02** · the splits | Where does quality vary: between countries, regions, cities, or providers? |
| **03** · multiple months | Is my country getting better or worse over time? |

## What is the Monthly Stats dataset?

[M-Lab](https://www.measurementlab.net/) runs an open speed-test platform used millions of times per day from real user devices worldwide. We aggregate those NDT results into **monthly, percentile-based summaries** at several geographic granularities.

Instead of querying millions or billions of raw rows of speed-test data in a database, each month you download one small file (roughly 1–5 MB) that already contains p1–p99 values for each metric, for each geography, for that month.

> The Internet Quality Barometer (IQB) project builds on this dataset — see the [IQB publications](https://www.measurementlab.net/publications/IQB_report_2025.pdf) for that composite-score view. Here we stay with the metrics themselves.

## The four metrics

| Metric | Column prefix | Unit | Better connectivity |
|---|---|---|---|
| Download throughput | `download_p*` | Mbit/s | ↑ Higher |
| Upload throughput | `upload_p*` | Mbit/s | ↑ Higher |
| Latency (min RTT) | `latency_p*` | ms | ↓ Lower |
| Packet loss rate | `loss_p*` | fraction 0–1 | ↓ Lower |

Plain English: **download** is how fast pages and files arrive, **upload** is how fast you send, **latency** is responsiveness (video calls, gaming), **loss** is dropped packets (reliability).

> **Percentile polarity** — for latency and loss, *lower* is better, so the data **flip the percentiles**: a higher percentile always means a *better* connection. `latency_p95` is the 5% of connections with the lowest (best) latency; `latency_p5` is the slowest (worst) latency. Notebook 01 makes this visible on real curves.

## Percentiles — line everyone up

A percentile is not complicated. **Line all the tests up in a month, slowest to fastest:**

- **p50** (the median) is the test in the exact middle — the typical user's experience;
- **p95** is 95 tests out of every 100 — near the front of the line;
- **p1** is near the very back.

There is no mean (average) in this dataset — only percentiles. That is a good thing: one very fast connection cannot drag the summary upward the way it drags an average.

A **wide p50→p95 gap is not automatically "inequality"** — it is a statement about the *shape* of the distribution. Most tests sit in a band, and a minority run much faster or much slower. Where the median sits, how long the tail stretches, how flat or steep the middle is: that shape is what notebook 01 teaches you to read.

And before believing any of it: **ask how many tests went in.** 100 tests wiggle; 100 000 are solid. Every ranking chart in this workshop shows its test count. Sample count may be the single most important idea for small countries — their numbers are real, but a month with few tests should not be over-read.

## The splits — which slices exist

"One row per what" is the only thing that changes between slices; every slice keeps the same four metrics and the same percentiles.

| Slice (prefix) | One row per… | What you can learn with it | Taught in |
|---|---|---|---|
| `by_country` | country | Cross-national benchmarking; how distributions differ between countries | 01 |
| `by_country_subdivision1` | state / province | Geography of quality *within* a country (urban vs rural) | 02 |
| `by_country_city` | city | City-level benchmarking; which cities stand out from their country | 02 |
| `by_country_asn` | provider (ASN) | Provider differences within a country — **handle with care: the data are not cleaned, so we compare, never rank or promote** | 02 |

Upload files mirror the download files: `uploads_by_country`, `uploads_by_country_city`, and so on.

## Dates and coverage

Slices are published asynchronously — at any moment the newest month can differ between slices by 1–2 months. Every notebook in this workshop derives "the newest month" from the manifest itself, and notebook 02 uses the newest month that exists in *all* the slices it needs. You never type a date by hand.

## How these notebooks load data

One pattern, everywhere in this folder:

```
manifest → pick month and slice → pd.read_parquet(url)
```

No local cache, no build step. Each file is ~1–5 MB, so a fresh download per run is fast and every notebook stays self-contained. (If you ever pull dozens of months repeatedly, a local cache is the optimization to reach for — not needed for this workshop.)

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import requests
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

print("Ready.")

In [ ]:
MANIFEST_URL = "https://measurementlab.net/data/stats/manifest.json"
manifest = requests.get(MANIFEST_URL, timeout=30).json()

records = []
for path, meta in manifest["files"].items():
    parts = path.split("/")
    # cache/v1/{start_ts}/{end_ts}/{slice_name}/data.parquet
    if len(parts) == 6 and parts[5] == "data.parquet":
        records.append({
            "start": pd.to_datetime(parts[2], format="%Y%m%dT%H%M%SZ"),
            "end":   pd.to_datetime(parts[3], format="%Y%m%dT%H%M%SZ"),
            "slice": parts[4],
            "url":   meta["url"],
        })

catalog = (pd.DataFrame(records)
           .sort_values(["slice", "start"])
           .reset_index(drop=True))


def month_url(slice_name, start):
    # Direct URL of one month of one slice. start: 'YYYY-MM-DD' (first of month).
    row = catalog[(catalog["slice"] == slice_name) &
                  (catalog["start"] == pd.to_datetime(start))]
    if row.empty:
        raise ValueError(f"No {slice_name} file for {start}")
    return row.iloc[0]["url"]


def latest_month(slice_name):
    # Newest month that exists for a slice, as 'YYYY-MM-DD'.
    return catalog.loc[catalog["slice"] == slice_name, "start"].max().strftime("%Y-%m-%d")

print("Catalog loaded —", len(catalog), "files,",
      catalog["slice"].nunique(), "slices,",
      catalog["start"].min().date(), "→", catalog["start"].max().date())

## Look at a sample of the data

Load the newest month of `downloads_by_country` — always derived from the manifest, never hardcoded — and look at the first rows.

In [ ]:
MONTH = latest_month("downloads_by_country")
df = pd.read_parquet(month_url("downloads_by_country", MONTH))

print("Newest month:", MONTH)
print("Rows:", len(df), "countries")
print()
print("Columns:", list(df.columns))
print()
df.head()

## Something to try — three tiny recipes

Three one-liners, adapted from an exploratory notebook by Pavlos. Each is the kernel of a much bigger question.

1. The median download for one country, for the newest month.
2. A table comparing several countries on two metrics at once.
3. Filtering a city slice to one city — the same pattern at finer resolution.

In [ ]:
# 1. One country, one number
us = df[df["country_code"] == "US"]["download_p50"].iloc[0]
print(f"US median download, {MONTH}: {us:.1f} Mbit/s")

# 2. A small comparison table
(df[df["country_code"].isin(["US", "DE", "BR", "IN", "NG"])]
    [["country_code", "download_p50", "latency_p50"]]
    .sort_values("download_p50", ascending=False))

In [ ]:
# 3. Filter a city slice to one city
city_month = latest_month("downloads_by_country_city")
city = pd.read_parquet(month_url("downloads_by_country_city", city_month))
london = city[city["city"] == "London"]
print(f"London, {city_month}: {len(london)} rows, "
      f"median download {london['download_p50'].iloc[0]:.0f} Mbit/s "
      f"({london['sample_count'].iloc[0]:,} tests)")

## Explore the catalog

Pick a slice, then a month: the cell prints the exact download URL and a preview of the first rows. This is the menu for everything you can ask the data.

In [ ]:
w_slice = widgets.Dropdown(
    options=sorted(catalog["slice"].unique()), value="downloads_by_country",
    description="Slice:", layout=widgets.Layout(width="420px"))
w_month = widgets.Dropdown(description="Month:",
                           layout=widgets.Layout(width="320px"))
out = widgets.Output()


def on_slice(change=None):
    months = (catalog[catalog["slice"] == w_slice.value]["start"]
              .dt.strftime("%Y-%m-%d").tolist())
    w_month.options = sorted(months, reverse=True)
    w_month.value = months[-1]


def show(change=None):
    with out:
        clear_output(wait=True)
        url = month_url(w_slice.value, w_month.value)
        print("Slice :", w_slice.value)
        print("Month :", w_month.value)
        print("URL   :", url)
        print()
        try:
            preview = pd.read_parquet(url).head()
            print(f"{len(preview)} of {pd.read_parquet(url).shape[0]:,} rows:")
            display(preview)
        except Exception as exc:
            print(f"Could not preview: {exc}")


w_slice.observe(on_slice, "value")
w_month.observe(show, "value")
display(widgets.VBox([w_slice, w_month, out]))
on_slice(); show()

## Next steps

- **01 · Country explorer** — read one country's distribution shape; see whether rankings hold at p95.
- **02 · The splits** — regions, cities, and providers inside a country.
- **03 · Multiple months** — trends over time, loaded the same direct way.